In [341]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

In [342]:
stats_df = pd.read_csv('..\\data\\raw\\player-stats.csv', index_col=0)

# 1. Limpeza e tratamento dos dados

#### 1.1 Colunas nulas e duplicatas de jogador

In [343]:
stats_df.isnull().sum()

Player      0
Nation      0
Pos         0
Squad       0
Age         0
Born        0
MP          0
Starts      0
Min         0
90s         0
Gls         0
Ast         0
G+A         0
G-PK        0
PK          0
PKatt       0
CrdY        0
CrdR        0
2CrdY       0
Fls         0
Fld         0
Off         0
Crs         0
Int         0
TklW        0
PKwon     761
PKcon     761
OG          0
dtype: int64

In [344]:
stats_df = stats_df.drop(columns=['PKwon', 'PKcon'])

In [345]:
stats_df[stats_df[['Player', 'Nation', 'Pos', 'Born']].duplicated(keep=False)]

,Player,Nation,Pos,Squad,Age,Born,MP,Starts,Min,90s,...,CrdY,CrdR,2CrdY,Fls,Fld,Off,Crs,Int,TklW,OG
Rk,,,,,,,,,,,,,,,,,,,,,
92,Pedro Borges,br BRA,MF,Fortaleza,27,1997,1,0,37,0.4,...,2,1,1,3,0,0,0,0,1,0
93,Pedro Borges,br BRA,MF,Sport Recife,27,1997,13,4,508,5.6,...,5,0,0,11,11,0,0,7,9,0
168,Lautaro Díaz,ar ARG,FW,Santos,26,1998,15,12,1020,11.3,...,2,0,0,17,9,5,7,3,2,0
169,Lautaro Díaz,ar ARG,FW,Cruzeiro,26,1998,6,0,85,0.9,...,0,0,0,0,0,0,2,0,0,0
186,Carlos Eduardo,br BRA,FWMF,Mirassol,28,1996,24,5,734,8.2,...,2,0,0,18,24,5,27,5,8,0
187,Carlos Eduardo,br BRA,FWMF,Vitória,28,1996,2,0,41,0.5,...,0,0,0,0,1,0,3,1,0,0
236,Gabriel,br BRA,GK,Vitória,32,1992,1,1,90,1.0,...,0,0,0,0,1,0,0,0,0,0
238,Gabriel,br BRA,GK,Sport Recife,32,1992,23,23,2070,23.0,...,3,0,0,0,4,0,0,0,0,0
297,Victor Hugo,br BRA,MF,Santos,20,2004,9,6,443,4.9,...,0,0,0,3,3,0,7,2,2,0


#### 1.2 Agregando linhas de jogadores iguais que jogaram por dois times diferentes durante a temporada

In [346]:
# Tirando as colunas que envolvam algum tipo de cálculo

stats_df = stats_df.drop(columns=['Age', '90s', 'G+A', 'G-PK'])

In [347]:
agg_dict = {
    'Player': 'first',
    'Nation': 'first',
    'Pos': 'first',
    'Squad': lambda x: ' / '.join(x.unique()),
    'Born': 'first',
    'MP': 'sum',
    'Starts': 'sum',
    'Min': 'sum',
    'Gls': 'sum',
    'Ast': 'sum',
    'PK': 'sum',
    'PKatt': 'sum',
    'CrdY': 'sum',
    'CrdR': 'sum',
    '2CrdY': 'sum',
    'Fls': 'sum',
    'Fld': 'sum',
    'Off': 'sum',
    'Crs': 'sum',
    'Int': 'sum',
    'TklW': 'sum',
    'OG': 'sum',
}

In [348]:
stats_df = stats_df.groupby(['Player', 'Nation', 'Pos', 'Born'], as_index=False).agg(agg_dict)

In [349]:
stats_df[stats_df[['Player', 'Nation', 'Pos', 'Born']].duplicated(keep=False)]

,Player,Nation,Pos,Squad,Born,MP,Starts,Min,Gls,Ast,...,CrdY,CrdR,2CrdY,Fls,Fld,Off,Crs,Int,TklW,OG


In [350]:
stats_df[stats_df['Player'] == 'Juan Sforza']['Squad']

375    Juventude / Vasco da Gama
Name: Squad, dtype: str

#### 1.3 Aplicar filtro de minutagem mínima para os jogadores

In [351]:
# Deixei a minutagem em mínima em 180 que é equivalente a 2 partidas completas de 90 minutos.
mediana_antes = stats_df['Min'].median()
print(f'Minutagem mediana antes do filtro de minutos: {mediana_antes}')

stats_df = stats_df[stats_df['Min'] > 180]

mediana_depois = stats_df['Min'].median()
print(f'\nMinutagem mediana depois do filtro de minutos: {mediana_depois}')

Minutagem mediana antes do filtro de minutos: 825.0

Minutagem mediana depois do filtro de minutos: 1046.0


#### 1.3 Recalculando 90s e exportando o dataset limpo

In [352]:
### Refazendo as colunas de cálculo (principalmente 90s) para que posteriomente eu consiga fazer o cálculo de G/90, A/90, etc.

stats_df['90s'] = (stats_df['Min'] / 90).round(1)

In [353]:
stats_df.to_csv('..\\data\\processed\\player-stats-clean.csv', index=False)  

### 2. Construindo a matriz de features para MLP e SOM

#### 2.1 Construindo o alvo de 5 classes

In [354]:
pos_map = {
    'DFMF': 'DF',
    'MFDF': 'MF',
    'FWMF': 'FW',
}

In [355]:
stats_df['Pos'] = stats_df['Pos'].replace(pos_map)
stats_df['Pos'].value_counts()

Pos
MF      219
DF      198
FW       84
MFFW     61
GK       44
Name: count, dtype: int64

#### 2.2 Convertendo as colunas de features para taxas por 90 minutos

In [356]:
features = ['Gls', 'Ast', 'CrdY', 'Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW']

# Eu tirei as colunas PK, PKatt, 2CrdY, OG, CrdR por conta do diagnóstico de skew (skew > 3) obtido no notebook target-kmeans-clustering.ipynb.

stats_df[features] = stats_df[features].div(stats_df['90s'], axis=0).round(2)

#### 2.3 Dropando as colunas que não serão usadas no treinamento

In [357]:
stats_df = stats_df.drop(columns=['Nation', 'Squad', 'Player', 'Born', 'Min', 'MP', 'Starts', 'PK', 'PKatt', '2CrdY', 'OG', 'CrdR', '90s'])

In [358]:
print(stats_df.columns)

print(stats_df.shape)

Index(['Pos', 'Gls', 'Ast', 'CrdY', 'Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW'], dtype='str')
(606, 10)


#### 2.4 Codificando o target com OneHotEncoder

In [359]:
encoder = OneHotEncoder(sparse_output=False)

In [360]:
pos_encoded = encoder.fit_transform(stats_df[['Pos']])

pos_classes = encoder.categories_[0]
pos_classes

array(['DF', 'FW', 'GK', 'MF', 'MFFW'], dtype=object)

In [361]:
pos_encoded_df = pd.DataFrame(pos_encoded, columns=pos_classes, index=stats_df.index)

stats_df = pd.concat([pos_encoded_df, stats_df], axis=1)

stats_df.head()

,DF,FW,GK,MF,MFFW,Pos,Gls,Ast,CrdY,Fls,Fld,Off,Crs,Int,TklW
0,1.0,0.0,0.0,0.0,0.0,DF,0.00,0.07,0.21,1.10,0.34,0.07,0.00,1.23,0.75
1,0.0,1.0,0.0,0.0,0.0,FW,0.71,0.20,0.71,3.27,2.35,0.20,0.20,0.00,1.02
2,0.0,1.0,0.0,0.0,0.0,FW,0.09,0.27,0.32,1.23,2.37,0.27,5.02,0.23,1.05
4,1.0,0.0,0.0,0.0,0.0,DF,0.00,0.00,0.49,1.57,1.08,0.10,2.75,1.37,0.69
5,1.0,0.0,0.0,0.0,0.0,DF,0.00,0.00,0.16,1.45,0.48,0.00,0.16,1.37,0.81


In [362]:
stats_df.to_csv('..\\data\\processed\\player-stats-model.csv', index=False)